In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import LabelEncoder

# High-cornering circuits by F1 domain knowledge
HIGH_CORNERING = {'Hungary', 'Spain', 'Japan', 'Austria', 'Great Britain',
                  'Netherlands', 'Bahrain', 'Singapore'}

df['CircuitType'] = df['Race'].apply(
    lambda x: 'High-Cornering' if x in HIGH_CORNERING else 'Standard'
)

# --- Build strategy table: one row per driver-race stint ---
stints = (
    df.groupby(['Race', 'Year', 'Driver', 'Stint', 'Compound', 'CircuitType'])
    .agg(StintLaps=('TyreLife', 'count'))
    .reset_index()
    .sort_values(['Race', 'Year', 'Driver', 'Stint'])
)

# Label each stint's position within the race (1st, 2nd, 3rd+)
stints['StintNumber'] = stints.groupby(['Race', 'Year', 'Driver'])['Stint'].rank(method='dense').astype(int)
stints['StintNumber'] = stints['StintNumber'].clip(upper=3).map({1: 'Stint 1', 2: 'Stint 2', 3: 'Stint 3+'})

# --- Model: predict Compound from CircuitType + StintNumber ---
model_df = stints[['CircuitType', 'StintNumber', 'Compound']].dropna()

le_circuit = LabelEncoder()
le_stint   = LabelEncoder()
le_compound = LabelEncoder()

X = pd.DataFrame({
    'CircuitType' : le_circuit.fit_transform(model_df['CircuitType']),
    'StintNumber' : le_stint.fit_transform(model_df['StintNumber']),
})
y = le_compound.fit_transform(model_df['Compound'])

tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y)
print('Decision tree rules:')
print(export_text(tree, feature_names=['CircuitType', 'StintNumber']))

# --- Recommendation function ---
def recommend_compound(circuit, stint_number):
    ct  = le_circuit.transform([circuit])[0]
    sn  = le_stint.transform([f'Stint {min(stint_number, 3)}' if stint_number < 3 else 'Stint 3+'])[0]
    pred = tree.predict([[ct, sn]])[0]
    proba = tree.predict_proba([[ct, sn]])[0]
    compound = le_compound.inverse_transform([pred])[0]
    confidence = proba.max()
    return compound, confidence

print('\nStrategy Recommendations:')
for circuit_type in ['High-Cornering', 'Standard']:
    print(f'\n  {circuit_type}:')
    for stint in [1, 2, 3]:
        compound, conf = recommend_compound(circuit_type, stint)
        print(f'    Stint {stint}: {compound}  ({conf*100:.0f}% of historical races)')

# --- Visualisation: compound usage by stint and circuit type ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
compound_colors = {'SOFT': '#e63946', 'MEDIUM': '#f4a261', 'HARD': '#457b9d'}

for ax, circuit_type in zip(axes, ['High-Cornering', 'Standard']):
    sub = stints[stints['CircuitType'] == circuit_type]
    pct = (
        sub.groupby(['StintNumber', 'Compound'])
        .size()
        .groupby(level=0)
        .transform(lambda x: x / x.sum() * 100)
        .reset_index(name='Pct')
    )
    pivot = pct.pivot(index='StintNumber', columns='Compound', values='Pct').fillna(0)
    pivot = pivot.reindex(index=['Stint 1', 'Stint 2', 'Stint 3+'],
                          columns=['SOFT', 'MEDIUM', 'HARD'], fill_value=0)
    pivot.plot(kind='bar', ax=ax, color=list(compound_colors.values()),
               edgecolor='white', width=0.7)
    ax.set_title(f'{circuit_type} Circuits')
    ax.set_xlabel('Stint')
    ax.set_ylabel('% of stints')
    ax.set_ylim(0, 100)
    ax.set_xticklabels(pivot.index, rotation=0)
    ax.legend(title='Compound')

plt.suptitle('Compound Choice by Stint & Circuit Type (2024–2025)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_08_strategy_model.png', bbox_inches='tight')
plt.show()


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Train on 2024, test on 2025
train = df[df['Year'] == 2024].dropna(subset=['TyreLife', 'CompoundEncoded', 'LapTimeDelta'])
test  = df[df['Year'] == 2025].dropna(subset=['TyreLife', 'CompoundEncoded', 'LapTimeDelta'])

X_train, y_train = train[['TyreLife', 'CompoundEncoded']], train['LapTimeDelta']
X_test,  y_test  = test[['TyreLife', 'CompoundEncoded']],  test['LapTimeDelta']

model  = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f} s')
print(f'R²  : {r2_score(y_test, y_pred):.3f}')
print(f'Coefficients — TyreLife: {model.coef_[0]:+.4f} s/lap | CompoundEncoded: {model.coef_[1]:+.4f} s/step')

# Degradation lines per compound
fig, ax = plt.subplots(figsize=(7, 4))
compound_colors = {'SOFT': '#e63946', 'MEDIUM': '#f4a261', 'HARD': '#457b9d'}

for compound, color in compound_colors.items():
    enc = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2}[compound]
    x   = np.linspace(1, 30, 100)
    y   = model.predict(pd.DataFrame({'TyreLife': x, 'CompoundEncoded': enc}))
    ax.plot(x, y, color=color, linewidth=2, label=compound)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Tyre Life (laps)')
ax.set_ylabel('Predicted LapTimeDelta (s)')
ax.set_title('Baseline Linear Model — Predicted Degradation by Compound')
ax.legend(title='Compound')
plt.tight_layout()
plt.savefig('plot_07_baseline_model.png', bbox_inches='tight')
plt.show()


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 5. Baseline Model

FEATURES = ['TyreLife', 'CompoundEncoded', 'TrackTemp', 'StintLength']
TARGET   = 'LapTimeDelta'

# Build modeling dataframe — drop rows with missing TrackTemp
model_df = df[FEATURES + [TARGET, 'Race', 'Year']].dropna().copy()

# One-hot encode circuit
circuit_dummies = pd.get_dummies(model_df['Race'], prefix='circuit', drop_first=True)
model_df = pd.concat([model_df.drop(columns=['Race']), circuit_dummies], axis=1)

feature_cols = [c for c in model_df.columns if c not in [TARGET, 'Year']]

# Temporal split: train on 2024, validate on 2025
train = model_df[model_df['Year'] == 2024]
test  = model_df[model_df['Year'] == 2025]

X_train, y_train = train[feature_cols], train[TARGET]
X_test,  y_test  = test[feature_cols],  test[TARGET]

# Fit
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f'Baseline Linear Regression — 2025 holdout')
print(f'  Train : {len(X_train):,} laps (2024)')
print(f'  Test  : {len(X_test):,} laps  (2025)')
print(f'  RMSE  : {rmse:.3f} s  (target < 0.300 s)')
print(f'  MAE   : {mae:.3f} s')
print(f'  R²    : {r2:.3f}')

# --- Visualize results ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Predicted vs actual (sample 2000 points to avoid overplotting)
sample = np.random.choice(len(y_test), size=min(2000, len(y_test)), replace=False)
axes[0].scatter(y_test.iloc[sample], y_pred[sample],
                alpha=0.3, s=8, color='#4c72b0')
lims = [max(y_test.min(), -5), min(y_test.max(), 10)]
axes[0].plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')
axes[0].set_xlabel('Actual LapTimeDelta (s)')
axes[0].set_ylabel('Predicted LapTimeDelta (s)')
axes[0].set_title(f'Predicted vs Actual  (RMSE={rmse:.3f}s)')
axes[0].legend()

# Feature coefficients (non-circuit features only)
core_features = FEATURES
coefs = pd.Series(
    model.coef_[:len(core_features)],
    index=core_features
).sort_values()

colors = ['#e63946' if v > 0 else '#457b9d' for v in coefs]
axes[1].barh(coefs.index, coefs.values, color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Coefficient (s per unit)')
axes[1].set_title('Feature Coefficients')

plt.tight_layout()
plt.savefig('plot_07_baseline_model.png', bbox_inches='tight')
plt.show()
